# Ride-hailing rider churn: decision tree

This notebook builds an interpretable binary classifier whose output is **Yes** (the rider is predicted to churn) or **No**. It is designed to run end-to-end on Kaggle.

The notebook first looks for a compatible CSV under `/kaggle/input`. If none is attached, it creates a reproducible synthetic ride-hailing dataset so every cell remains runnable.

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay, classification_report, roc_auc_score
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree

warnings.filterwarnings('ignore', category=FutureWarning)
RANDOM_STATE = 42
TARGET = 'churned'

## 2. Load data

A compatible CSV must contain a target named `churned`, `churn`, `is_churned`, or `target`. The target may be 0/1, Yes/No, True/False, or Churned/Active. To avoid target leakage, identifiers and columns that directly reveal the outcome are removed later.

In [ ]:
TARGET_ALIASES = ['churned', 'churn', 'is_churned', 'target']

def find_compatible_csv(root='/kaggle/input'):
    root = Path(root)
    if not root.exists():
        return None, None
    for path in sorted(root.rglob('*.csv')):
        try:
            sample = pd.read_csv(path, nrows=5)
        except Exception:
            continue
        lower_to_original = {str(c).strip().lower(): c for c in sample.columns}
        for alias in TARGET_ALIASES:
            if alias in lower_to_original:
                return path, lower_to_original[alias]
    return None, None

def make_demo_data(n=5000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({
        'rider_id': np.arange(1, n + 1),
        'city': rng.choice(['Tallinn', 'Riga', 'Vilnius', 'Warsaw'], n),
        'tenure_months': rng.integers(1, 49, n),
        'trips_last_30d': rng.poisson(7, n),
        'days_since_last_trip': rng.integers(0, 61, n),
        'avg_fare_eur': np.round(rng.normal(12, 3, n).clip(3), 2),
        'promo_share_90d': np.round(rng.beta(2, 5, n), 3),
        'cancel_rate_90d': np.round(rng.beta(1.5, 8, n), 3),
        'support_contacts_90d': rng.poisson(0.7, n),
        'avg_pickup_eta_min': np.round(rng.normal(6.5, 2, n).clip(1), 1),
        'payment_type': rng.choice(['card', 'cash', 'wallet'], n, p=[0.72, 0.18, 0.10]),
    })
    logit = (
        -2.5 + 0.075 * df['days_since_last_trip']
        - 0.18 * df['trips_last_30d'] + 1.8 * df['cancel_rate_90d']
        + 0.14 * df['support_contacts_90d'] + 0.08 * (df['avg_pickup_eta_min'] - 6)
        + 0.45 * (df['payment_type'] == 'cash') - 0.01 * df['tenure_months']
    )
    probability = 1 / (1 + np.exp(-logit))
    df[TARGET] = rng.binomial(1, probability)
    return df

csv_path, source_target = find_compatible_csv()
if csv_path is None:
    df = make_demo_data()
    print('No compatible CSV found; using reproducible synthetic demo data.')
else:
    df = pd.read_csv(csv_path).rename(columns={source_target: TARGET})
    print(f'Loaded: {csv_path}')

print(f'Shape: {df.shape}')
display(df.head())

## 3. Validate target and prepare features

All feature selection happens before the train/test split, but no statistics are learned from the full dataset: imputers and one-hot encoding are fitted inside the pipeline on training folds only.

In [ ]:
def normalize_binary_target(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype(int)
    if pd.api.types.is_numeric_dtype(series):
        values = set(series.dropna().unique())
        if values.issubset({0, 1}):
            return series.astype('Int64')
    mapping = {
        'yes': 1, 'y': 1, 'true': 1, 'churned': 1, 'churn': 1, '1': 1,
        'no': 0, 'n': 0, 'false': 0, 'active': 0, 'retained': 0, '0': 0,
    }
    normalized = series.astype(str).str.strip().str.lower().map(mapping)
    if normalized.isna().any():
        unknown = sorted(series[normalized.isna()].astype(str).unique())[:10]
        raise ValueError(f'Target must be binary. Unrecognized values: {unknown}')
    return normalized.astype(int)

df = df.copy()
df[TARGET] = normalize_binary_target(df[TARGET])
df = df.dropna(subset=[TARGET])
if df[TARGET].nunique() != 2:
    raise ValueError('The target must contain both churned and non-churned riders.')

# IDs add no generalizable signal; outcome-derived columns would leak the answer.
leakage_names = {'churn_date', 'account_closed', 'deactivated_at', 'days_after_churn'}
id_columns = [c for c in df.columns if c.lower() == 'id' or c.lower().endswith('_id')]
leakage_columns = [c for c in df.columns if c.lower() in leakage_names]
drop_columns = sorted(set(id_columns + leakage_columns))

X = df.drop(columns=[TARGET] + drop_columns)
y = df[TARGET].astype(int)
if X.shape[1] == 0:
    raise ValueError('No usable feature columns remain after removing target/ID columns.')

print(f'Churn rate: {y.mean():.1%}')
print(f'Dropped ID/leakage columns: {drop_columns or None}')
display(y.value_counts().rename(index={0: 'No', 1: 'Yes'}).to_frame('riders'))

## 4. Split, preprocess, and tune the tree

The held-out test set is untouched during tuning. Five-fold cross-validation selects tree depth, minimum leaf size, and class weighting using ROC AUC.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

numeric_features = X.select_dtypes(include='number').columns.tolist()
categorical_features = X.columns.difference(numeric_features).tolist()

numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median'))])
categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ('num', numeric_pipe, numeric_features),
    ('cat', categorical_pipe, categorical_features),
])
pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', DecisionTreeClassifier(random_state=RANDOM_STATE)),
])

param_grid = {
    'model__max_depth': [3, 5, 7, None],
    'model__min_samples_leaf': [10, 25, 50],
    'model__class_weight': [None, 'balanced'],
}
search = GridSearchCV(
    pipeline, param_grid, scoring='roc_auc', cv=5, n_jobs=-1, refit=True
)
search.fit(X_train, y_train)
model = search.best_estimator_
print('Best parameters:', search.best_params_)
print(f'Best cross-validation ROC AUC: {search.best_score_:.3f}')

## 5. Evaluate on unseen riders

In [ ]:
y_pred = model.predict(X_test)
y_probability = model.predict_proba(X_test)[:, 1]

print(f'Test ROC AUC: {roc_auc_score(y_test, y_probability):.3f}')
print(classification_report(y_test, y_pred, target_names=['No', 'Yes'], digits=3))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=['No', 'Yes'], cmap='Blues', colorbar=False
)
plt.title('Churn confusion matrix')
plt.show()

## 6. Explain the model

Feature importance shows which inputs the fitted tree uses most. The plotted top levels expose the actual Yes/No rules.

In [ ]:
feature_names = model.named_steps['preprocess'].get_feature_names_out()
tree_model = model.named_steps['model']
importance = (
    pd.Series(tree_model.feature_importances_, index=feature_names, name='importance')
    .sort_values(ascending=False)
    .head(15)
)
display(importance.to_frame())
importance.sort_values().plot.barh(figsize=(8, 5), title='Top feature importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

plt.figure(figsize=(22, 10))
plot_tree(
    tree_model, feature_names=feature_names, class_names=['No', 'Yes'],
    filled=True, rounded=True, max_depth=3, fontsize=8
)
plt.title('Decision tree (first four levels)')
plt.show()

## 7. Produce the required Yes/No output

`predictions` contains one row per held-out rider. `churn_prediction` is the requested business output; probability is included only as useful supporting information.

In [ ]:
predictions = X_test.copy()
predictions['actual_churn'] = y_test.map({0: 'No', 1: 'Yes'})
predictions['churn_prediction'] = pd.Series(y_pred, index=X_test.index).map({0: 'No', 1: 'Yes'})
predictions['churn_probability'] = np.round(y_probability, 4)
display(predictions.head(20))

output_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
output_path = output_dir / 'churn_decision_tree_predictions.csv'
predictions.to_csv(output_path, index=False)
print(f'Saved predictions to {output_path}')

### Using this with real data

Attach a Kaggle dataset containing a compatible binary target and rerun all cells. Keep only features available at the moment the prediction is made. In particular, do not include account-closure fields, future trips, or activity measured after the churn observation window.